# IEMS 490 Wave Training
Trains an open source version of Google's Wavenet architecture to fit our 'waves' from our LaserPowerCurrent and SignalPdInGaAs variables

In [1]:
import torch
from torch_fn.wavenet_lstm import lr_schedule, WaveNet_LSTM #LSTM of WaveNet
from torch_fn.wavenet import WaveNet # Normal WaveNet, likely from the paper. Need someone who is literate to check.
import h5py
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import torch.nn as nn
import torch.optim as optim

### For each time step in each file, get the Laser Power and Signal
We don't want to use X, Y because that can change in each layer

In [2]:
folder_path = './DATASET' # Replace with your folder path
# This creates a list of extensions (including the dot)
file_paths = [f for f in Path(folder_path).iterdir() if f.is_file() and f.suffix == '.hdf5']

print(file_paths)

[WindowsPath('DATASET/layer1.hdf5'), WindowsPath('DATASET/layer10.hdf5'), WindowsPath('DATASET/layer100.hdf5'), WindowsPath('DATASET/layer101.hdf5'), WindowsPath('DATASET/layer102.hdf5'), WindowsPath('DATASET/layer103.hdf5'), WindowsPath('DATASET/layer104.hdf5'), WindowsPath('DATASET/layer105.hdf5'), WindowsPath('DATASET/layer106.hdf5'), WindowsPath('DATASET/layer107.hdf5'), WindowsPath('DATASET/layer108.hdf5'), WindowsPath('DATASET/layer109.hdf5'), WindowsPath('DATASET/layer11.hdf5'), WindowsPath('DATASET/layer110.hdf5'), WindowsPath('DATASET/layer111.hdf5'), WindowsPath('DATASET/layer112.hdf5'), WindowsPath('DATASET/layer113.hdf5'), WindowsPath('DATASET/layer114.hdf5'), WindowsPath('DATASET/layer115.hdf5'), WindowsPath('DATASET/layer116.hdf5'), WindowsPath('DATASET/layer117.hdf5'), WindowsPath('DATASET/layer118.hdf5'), WindowsPath('DATASET/layer119.hdf5'), WindowsPath('DATASET/layer12.hdf5'), WindowsPath('DATASET/layer120.hdf5'), WindowsPath('DATASET/layer121.hdf5'), WindowsPath('DAT

In [3]:
file1 = h5py.File(file_paths[0], 'r')
file1

<HDF5 file "layer1.hdf5" (mode r)>

In [30]:
xs, ys = file1['OpenData'][0], file1['OpenData'][1]
x_dict, y_dict = {}, {}
n=1
for i in range(len(xs)):
    if round(xs[i],n) not in x_dict.keys():
        x_dict[round(xs[i],n)] = 1
    else:
        x_dict[round(xs[i],n)] += 1
    if round(ys[i],n) not in y_dict.keys():
        y_dict[round(ys[i],n)] = 1
    else:
        y_dict[round(ys[i],n)] += 1
vals = []
for x_val in x_dict.keys():
    vals.append(x_dict[x_val])
print(len(vals))
vals = []
for y_val in y_dict.keys():
    vals.append(y_dict[y_val])
print(len(vals))

94
94


In [31]:
coords = []
for x in x_dict.keys():
    for y in y_dict.keys():
        # Set as tuples rather than lists because we don't need to alter afterwards
        coords.append((x,y))

#### Will take a long time (sad) [maybe 5 minutes]
Will save this as an .npz file bc we're not doing this again unless we're changing the coordinates

In [32]:
#data = []
lazer_data = []
signal_data = []
for file in file_paths:
    file_info = h5py.File(file, 'r')
    file_data = file_info['OpenData']
    xs, ys, lazer, signal = file_data[0], file_data[1], file_data[5], file_data[6]
    #x_dict, y_dict= {}, {}
    lazer_dict, signal_dict = {}, {}
    for i in range(len(xs)):
        new_coords = (round(xs[i], 1), round(ys[i],1))
        if new_coords not in lazer_dict.keys():
            lazer_dict[new_coords] = [lazer[i]]
            signal_dict[new_coords] = [signal[i]]
        else:
            lazer_dict[new_coords].append(lazer[i])
            #lazer_dict[new_coords][1]+=1
            signal_dict[new_coords].append(signal[i])
            #signal_dict[new_coords][1]+=1
    lazer_data.append(lazer_dict)
    signal_data.append(signal_dict)


In [33]:
median_lazer = []
median_signal = []
for i in range(len(lazer_data)):
    l_dict, s_dict = lazer_data[i], signal_data[i]
    
    # Process all coordinates for this file in one go
    batch_l = [np.median(l_dict[c]) if c in l_dict else 0 for c in coords]
    batch_s = [np.median(s_dict[c]) if c in s_dict else 0 for c in coords]
    
    median_lazer.append(batch_l)
    median_signal.append(batch_s)

In [41]:
np.savez('my_arrays.npz', arr_a=np.array(median_lazer), arr_b=np.array(median_signal))

### All data should now be the same length

In [ ]:
# Load the data from the .npz file
npz_file = np.load('my_arrays.npz')

# Access individual arrays
loaded_array_a = npz_file['arr_a']
loaded_array_b = npz_file['arr_b']

# Optional: check the names of all arrays in the file
print(f"Arrays in file: {npz_file.files}")

# Close the NpzFile object
npz_file.close()


In [ ]:
#path_tensors = [torch.tensor(np.stack([median_lazer[i], median_signal[i]], axis=0), dtype=torch.float32) 
#                for i in range(len(median_lazer))]

In [34]:
len(median_lazer)

379

In [ ]:
# Split and turn to tensor
#train_x, test_x = train_test_split(path_tensors, test_size=0.6)
train_x, test_x = train_test_split(median_lazer, test_size=0.6)

In [ ]:
#
# This errors because the dimensions are not consistent. There are three approaches, we can normalize the x and y and feed them as inputs and pad missing values
# OR we keep only lazer and signal and pad missing values OR we only sample ~300 points from each data sample
# 
x_train_tensor = torch.tensor(train_x)
x_test_tensor  = torch.tensor(test_x)

train_dataset = TensorDataset(x_train_tensor, x_train_tensor)
test_dataset  = TensorDataset(x_test_tensor, x_test_tensor)
batch_size = 1

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

RuntimeError: Could not infer dtype of dict

### Below code is buggy because padding is messed up

### We might be able to train BERT like encoder to fill in the blanks